In [7]:
# Dependencies
import numpy as np
import pandas as pd

def factorize_matrix(matrix: np.ndarray, rank: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Decomposes a matrix into two smaller matrices W and H using Singular Value Decomposition (SVD).

    Args:
        matrix (np.ndarray): The input matrix of shape (M, N).
        rank (int): The latent dimension or desired low rank (k).

    Returns:
        tuple: (W, H) where W has shape (M, k) and H has shape (k, N).
    """
    # Perform SVD
    # U: Left singular vectors, S: Singular values, Vt: Right singular vectors
    U, S, Vt = np.linalg.svd(matrix, full_matrices=False)

    # Truncate to the desired rank k to compress the parameter space
    W = U[:, :rank] @ np.diag(S[:rank])
    H = Vt[:rank, :]

    return W, H

In [8]:
# Step 1: Initialize the original matrix I (400x10) and factorize it
# Setting a fixed seed ensures the random initialization is reproducible across multiple runs
np.random.seed(42)

# Generate the initial weight matrix for the 400-dimensional input and 10-dimensional output
I_original = np.random.uniform(low=-1.0, high=1.0, size=(400, 10))

# Factorize I_original into A_original (400x3) and B_original (3x10)
A_original, B_original = factorize_matrix(I_original, rank=3)

print("--- SYSTEM INITIALIZATION ---")
print(f"Original Weight Matrix I shape: {I_original.shape}")
print(f"Low-rank Matrix A shape: {A_original.shape}")
print(f"Low-rank Matrix B shape: {B_original.shape}\n")
print(f"Total parameters reduced from {I_original.size} to {A_original.size + B_original.size}.")

--- SYSTEM INITIALIZATION ---
Original Weight Matrix I shape: (400, 10)
Low-rank Matrix A shape: (400, 3)
Low-rank Matrix B shape: (3, 10)

Total parameters reduced from 4000 to 1230.


In [9]:
# Step 2: Load the parameter changes (deltas) for A and B after training
# Note: If your local files are strictly Excel formats (.xlsx), comment out the read_csv lines
# and use pd.read_excel('filename.xlsx', header=None).to_numpy() instead.
excel_file_path = 'Demo Matrix Factorization.xlsx'
sheet_name_A = 'Matrix A'
sheet_name_B = 'Matrix B'

try:
    # Load the trained parameter updates (assuming no header row in the data files)
    A_changes = pd.read_excel(excel_file_path, sheet_name=sheet_name_A, header=None).to_numpy()
    B_changes = pd.read_excel(excel_file_path, sheet_name=sheet_name_B, header=None).to_numpy()

    print("--- LOADING WEIGHT UPDATES ---")
    print(f"Delta A shape: {A_changes.shape}")
    print(f"Delta B shape: {B_changes.shape}\n")

    # Step 3: Apply the gradient updates to the original low-rank matrices
    # This reflects the core concept of LoRA: W_new = W_0 + (Delta W)
    A_new = A_original + A_changes
    B_new = B_original + B_changes

    # Step 4: Reconstruct the final weight matrix I for the Neural Network inference
    I_reconstructed = A_new @ B_new

    print("--- RECONSTRUCTION RESULTS ---")
    print(f"Final Reconstructed Matrix I shape: {I_reconstructed.shape}")
    print("Sample output (First 3 rows of the updated Matrix I):")
    print(np.round(I_reconstructed[:3, :], 4))

except FileNotFoundError:
    print("Error: Dataset files not found. Please ensure the files are uploaded to the same directory as this Notebook.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

--- LOADING WEIGHT UPDATES ---
Delta A shape: (400, 3)
Delta B shape: (3, 10)

--- RECONSTRUCTION RESULTS ---
Final Reconstructed Matrix I shape: (400, 10)
Sample output (First 3 rows of the updated Matrix I):
[[-0.592   1.209   0.4423 -0.2131 -0.7736 -0.513  -0.4089  0.5408  0.4508
   0.1827]
 [-0.5425  1.207   0.5675 -0.2575 -0.8505 -0.2869 -0.544   0.5513  0.1177
   0.0367]
 [ 0.3162 -0.6276 -0.0498 -0.2305 -0.2803  0.2525 -0.0446 -0.0334 -0.4884
  -0.289 ]]


In [10]:
import numpy as np
import pandas as pd

# Set fixed seed
np.random.seed(42)

# Demo factorization #1: 10 x 8 into 10 x 3 and 3 x 8
# Generate an initial matrix_I with shape (400, 10) for factorization
# This assumes the full weight matrix before LoRA application is (400, 10)
matrix_I = np.random.uniform(low=-1.0, high=1.0, size=(400, 10))
print("Original matrix:")
print(matrix_I)

# Factorize matrix_I into matrix_A (400, 3) and matrix_B (3, 10)
# This will make matrix_B's shape consistent with delta_B's shape (3, 10)
matrix_A, matrix_B = factorize_matrix(matrix_I, 3)
print("Factorized result:")
print(matrix_A)
print(matrix_B)
print(matrix_A.shape)
print(matrix_B.shape)

try:
    # Read delta_A and delta_B from the Excel file (assuming they are correctly sized)
    excel_file = 'Demo Matrix Factorization.xlsx'
    delta_A = pd.read_excel(excel_file, sheet_name='Matrix A', header=None).to_numpy()
    delta_B = pd.read_excel(excel_file, sheet_name='Matrix B', header=None).to_numpy()

    print(f"delta_A shape: {delta_A.shape}")
    print(f"delta_B shape: {delta_B.shape}")

    if delta_A.shape != (400, 3) or delta_B.shape != (3, 10):
        print("CẢNH BÁO: Kích thước file Excel không đúng (cần 400x3 và 3x10 cho delta). Hãy kiểm tra lại file của bạn!")

    # Now matrix_A and delta_A, and matrix_B and delta_B should have compatible shapes
    A_new = matrix_A + delta_A
    B_new = matrix_B + delta_B

    I_new = A_new @ B_new
    print(I_new[0, :])

except FileNotFoundError as e:
    print("\n[LỖI]: Không tìm thấy file Excel!")
except ValueError as e:
    print(f"\n[LỖI DỮ LIỆU]: {e}")
except Exception as e:
    print(f"\n[LỖI]: An unexpected error occurred: {e}")

Original matrix:
[[-0.25091976  0.90142861  0.46398788 ...  0.73235229  0.20223002
   0.41614516]
 [-0.95883101  0.9398197   0.66488528 ...  0.04951286 -0.13610996
  -0.41754172]
 [ 0.22370579 -0.72101228 -0.4157107  ...  0.02846888  0.18482914
  -0.90709917]
 ...
 [ 0.94760083  0.08839733 -0.44952164 ...  0.09973713 -0.89885379
  -0.14775752]
 [ 0.66472708  0.61129761 -0.55136288 ... -0.09988272 -0.32509144
   0.74137691]
 [-0.83295017 -0.57774811  0.50493668 ... -0.21085537  0.05988117
  -0.67726528]]
Factorized result:
[[ 0.8006147   1.38193585  0.41705685]
 [ 1.16988165  1.18360697  0.27581501]
 [ 0.33196004 -0.79595606  0.09454697]
 ...
 [ 0.63258184 -0.89157304 -0.60205956]
 [-0.14539124 -0.01222281 -0.99901541]
 [ 0.57814869 -0.09316167  0.30806762]]
[[-0.0877642   0.14178326  0.34527838 -0.18657976 -0.61478274  0.15817805
  -0.33651108  0.17424989 -0.42421569 -0.30269785]
 [-0.31470042  0.71109097  0.25457388  0.04217809 -0.00528489 -0.21108168
  -0.16789779  0.09433474  0.4626

In [11]:
# Step 5: Evaluate Model Performance on a specific input image
# Goal: Calculate the Cross-Entropy loss for label 4 before and after applying the updates.

import numpy as np
import pandas as pd

def compute_stable_softmax(logits: np.ndarray) -> np.ndarray:
    """
    Applies the Softmax function to convert raw logits into a probability distribution.
    Includes a numerical stability trick (subtracting the max logit).
    """
    e_x = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

def compute_cross_entropy(probs: np.ndarray, target_label: int) -> float:
    """
    Computes the Categorical Cross-Entropy Loss (Negative Log-Likelihood) for the given target class.
    """
    # Adding a tiny epsilon (1e-15) prevents log(0) errors in edge cases
    return -np.log(probs[0, target_label] + 1e-15)

try:
    # 1. Load the input image
    # Note: Adjust filename if using pd.read_excel('...', header=None) instead of read_csv
    excel_file_path = 'Demo Matrix Factorization.xlsx'
    sheet_name_img = 'input-image'

    # Read the 400x1 column and transpose/reshape it to (1, 400) for batched matrix multiplication
    X = pd.read_excel(excel_file_path, sheet_name=sheet_name_img, header=None).to_numpy().reshape(1, 400)

    # 2. Define the ground-truth label based on the problem statement
    target_label = 4

    # 3. Calculate baseline weights and perform the forward pass BEFORE training
    W_original = A_original @ B_original
    logits_before = X @ W_original
    probs_before = compute_stable_softmax(logits_before)
    loss_before = compute_cross_entropy(probs_before, target_label)

    # 4. Calculate updated weights and perform the forward pass AFTER training
    W_new = A_new @ B_new
    logits_after = X @ W_new
    probs_after = compute_stable_softmax(logits_after)
    loss_after = compute_cross_entropy(probs_after, target_label)

    # 5. Output the comparative results
    print("--- EVALUATION RESULTS FOR LABEL = 4 ---")
    print("BEFORE TRAINING (Original Factorization):")
    print(f"  -> Predicted Probability: {probs_before[0, target_label] * 100:.4f}%")
    print(f"  -> Cross-Entropy Loss:    {loss_before:.4f}\n")

    print("AFTER TRAINING (Applying Delta Updates):")
    print(f"  -> Predicted Probability: {probs_after[0, target_label] * 100:.4f}%")
    print(f"  -> Cross-Entropy Loss:    {loss_after:.4f}")

except FileNotFoundError:
    print(f"Error: The Excel file '{excel_file_path}' or sheet '{sheet_name_img}' was not found. Please ensure it is in the correct directory.")
except NameError as ne:
    print(f"Initialization Error: Ensure the previous cells defining A_original, B_original, A_new, and B_new have been executed. Details: {ne}")
except Exception as e:
    print(f"An unexpected error occurred during evaluation: {e}")

--- EVALUATION RESULTS FOR LABEL = 4 ---
BEFORE TRAINING (Original Factorization):
  -> Predicted Probability: 7.0553%
  -> Cross-Entropy Loss:    2.6514

AFTER TRAINING (Applying Delta Updates):
  -> Predicted Probability: 20.5899%
  -> Cross-Entropy Loss:    1.5804
